In [ ]:
!pip install -q chromadb

In [ ]:
import chromadb

client = chromadb.Client()

# Create or reuse collection
collection = client.get_or_create_collection(name="crud_demo")

print("Collection created:", collection.name)
print("Current count:", collection.count())

In [ ]:
collection.add(
    ids=["doc1", "doc2", "doc3", "doc4", "doc5"],
    documents=[
        "Python is a programming language.",
        "Today the weather is sunny.",
        "Machine learning learns from data.",
        "I enjoy playing football.",
        "Deep learning uses neural networks."
    ],
    metadatas=[
        {"category": "tech"},
        {"category": "weather"},
        {"category": "tech"},
        {"category": "sports"},
        {"category": "tech"}
    ]
)

print("Documents added:", collection.count())

In [ ]:
preview = collection.peek(limit=3)

for id, doc in zip(preview["ids"], preview["documents"]):
    print(id, "->", doc)

In [ ]:
result = collection.get(ids=["doc1", "doc3"])

for id, doc, meta in zip(
    result["ids"],
    result["documents"],
    result["metadatas"]
):
    print(id, "|", meta, "|", doc)

In [ ]:
results = collection.query(
    query_texts=["Tell me about AI and neural networks"],
    n_results=2
)

for id, doc, distance in zip(
    results["ids"][0],
    results["documents"][0],
    results["distances"][0]
):
    print(id, "|", distance, "|", doc)

In [ ]:
results = collection.query(
    query_texts=["Something fun to do outside"],
    n_results=3,
    where={"category": "sports"}
)

for id, doc in zip(
    results["ids"][0],
    results["documents"][0]
):
    print(id, "->", doc)

In [ ]:
collection.update(
    ids=["doc4"],
    documents=["I enjoy playing football and basketball."],
    metadatas=[{"category": "sports", "updated": True}]
)

updated = collection.get(ids=["doc4"])

print(updated["documents"])
print(updated["metadatas"])

In [ ]:
# Add 5 new documents
collection.add(
    ids=["doc6", "doc7", "doc8", "doc9", "doc10"],
    documents=[
        "Python is used for data science.",
        "Java is an object-oriented language.",
        "SQL is used for database management.",
        "Machine learning predicts results.",
        "AI is transforming technology."
    ],
    metadatas=[
        {"category": "tech"},
        {"category": "programming"},
        {"category": "database"},
        {"category": "AI"},
        {"category": "tech"}
    ]
)

print("After adding 5:", collection.count())

# Update 2 documents and create 1 new document
collection.upsert(
    ids=["doc6", "doc7", "doc11"],
    documents=[
        "Python is widely used in AI.",
        "Java supports object-oriented programming.",
        "Deep learning is a part of machine learning."
    ],
    metadatas=[
        {"category": "AI"},
        {"category": "programming"},
        {"category": "AI"}
    ]
)

print("After upsert:", collection.count())
print(collection.get(ids=["doc6", "doc7", "doc11"]))

In [ ]:
# Delete all documents where category is AI
collection.delete(
    where={"category": "AI"}
)

# Confirm deletion
print("Remaining documents:", collection.count())
print("Remaining IDs:", collection.get()["ids"])

In [ ]:
def safe_add(collection, id, text):
    collection.upsert(
        ids=[id],
        documents=[text]
    )

# Add a new document
safe_add(collection, "doc12", "Natural language processing uses AI.")

# Update the same ID without duplicate error
safe_add(collection, "doc12", "NLP helps computers understand language.")

# Verify
print(collection.get(ids=["doc12"]))
print("Total documents:", collection.count())